# Ablation: Judge Models

Compares Qwen-2.5-14B vs Llama-3.1-8B as LLM judges on both Italian and English.
Measures inter-judge correlation and scoring bias.

**Memory strategy**: SigExt on CPU -> unloaded -> LLM for inference -> freed -> judge models swapped.

In [ ]:
import warnings, os
from dotenv import load_dotenv
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

!uv pip install -e ../..
load_dotenv()
from huggingface_hub import login
login(token=os.getenv('HF_TOKEN'))


In [ ]:
from sm_sip.config import SigExtConfig
from sm_sip.data import get_test_data
from sm_sip.models import load_sigext_model, unload_sigext_model, load_llm, create_summary_chain, create_judge_chain, preprocess_dataset
from sm_sip.prompts import get_summary_prompt, get_judge_prompt
from sm_sip.pipelines import run_inference
from sm_sip.metrics.judge import llm_judge_evaluate
from sm_sip.utils.io import save_results
from sm_sip.utils.gpu import clear_gpu_memory
import numpy as np

## Step 1: Generate summaries for both languages

In [ ]:
LANG_CONFIGS = {
    'it': {'preset': 'xlmr-5k-060t', 'quant': '8bit'},
    'en': {'preset': 'xlmr-5k-060t',  'quant': '8bit'},
}

inference_results = {}

for lang, lcfg in LANG_CONFIGS.items():
    print(f'\n{"="*60}\n  Generating summaries: {lang.upper()}\n{"="*60}')
    sc = SigExtConfig.from_preset(lang, lcfg['preset'])
    data = get_test_data(lang=lang, num_samples=30, skip_samples=sc.skip_samples)
    sm, st = load_sigext_model(sc.model_id, device='cpu')
    proc = preprocess_dataset(data, sm, st, lang=lang)
    unload_sigext_model(sm, st)

    _, _, pipe = load_llm('meta-llama/Llama-3.1-8B-Instruct', lcfg['quant'])
    chain = create_summary_chain(pipe, get_summary_prompt(lang, 'source_aware'))
    inference_results[lang] = run_inference(proc, chain)
    clear_gpu_memory()
    print(f'  Generated {len(inference_results[lang])} summaries for {lang.upper()}.')

## Step 2: Judge with different models

In [ ]:
JUDGE_MODELS = ['Qwen/Qwen2.5-14B-Instruct', 'meta-llama/Llama-3.1-8B-Instruct']
DIMENSIONS = ['faithfulness', 'completeness', 'conciseness', 'abstraction']

judge_results = {}

for jm in JUDGE_MODELS:
    jm_name = jm.split('/')[-1]
    print(f'\n{"="*60}\n  Judge: {jm_name}\n{"="*60}')
    _, _, jp = load_llm(jm, '8bit')
    jc = create_judge_chain(jp, get_judge_prompt('unified'))

    for lang, inf_res in inference_results.items():
        key = f'{lang}_{jm_name}'
        print(f'  -> Judging {lang.upper()} ({len(inf_res)} samples)...')
        scores = {d: [] for d in DIMENSIONS}
        for r in inf_res:
            s = llm_judge_evaluate(r['source'], r['generated_summary'], r['reference'], jc)
            for d in DIMENSIONS:
                scores[d].append(s.get(d, 3))
        judge_results[key] = {
            d: {'mean': float(np.mean(v)), 'std': float(np.std(v))}
            for d, v in scores.items()
        }

    clear_gpu_memory()

save_results({'ablation': 'judge_models', 'results': judge_results}, 'results/ablation_judge_models.json')
print('\nDone!')